# 🛒 TIỂU LUẬN MÔN BIG DATA – INSTACART ONLINE GROCERY
## PySpark Analysis: SparkSession + 20 Advanced SQL Queries + Dashboard
**Dataset:** [Instacart Online Grocery Basket Analysis Dataset – yasserh](https://www.kaggle.com/datasets/yasserh/instacart-online-grocery-basket-analysis-dataset)  
**Tech Stack:** Apache Spark (PySpark · Spark SQL · MLlib), Matplotlib, Seaborn  
**Course:** Big Data – UEH  

---
### 📋 Schema tổng quan (6 tables)
| Bảng | Mô tả | Rows (ước tính) | Cols |
|------|--------|----------------|------|
| `orders` | order_id, user_id, eval_set, order_number, order_dow, order_hour_of_day, days_since_prior_order | ~3,421,083 | 7 |
| `order_products__prior` | order_id, product_id, add_to_cart_order, reordered | ~32,434,489 | 4 |
| `order_products__train` | order_id, product_id, add_to_cart_order, reordered | ~1,384,617 | 4 |
| `products` | product_id, product_name, aisle_id, department_id | 49,688 | 4 |
| `aisles` | aisle_id, aisle | 134 | 2 |
| `departments` | department_id, department | 21 | 2 |


In [1]:
# Cài đặt PySpark trên Kaggle (chạy 1 lần)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'pyspark', 'findspark', '-q'],
               capture_output=True)
print('✅ pyspark & findspark ready')


✅ pyspark & findspark ready


In [4]:
import warnings
warnings.filterwarnings('ignore')

import os, findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import seaborn as sns

print('✅ Imports OK')


✅ Imports OK


## ⚡ PHẦN 1 — Thiết lập SparkSession


In [6]:
spark = (
    SparkSession.builder
    .appName('Instacart_BigData_Analysis')
    .master('local[*]')                              # Kaggle: 2 CPU
    .config('spark.driver.memory', '8g')
    .config('spark.executor.memory', '4g')
    .config('spark.sql.shuffle.partitions', '8')     # giảm overhead cho local
    .config('spark.default.parallelism', '4')
    .config('spark.sql.adaptive.enabled', 'true')    # AQE
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true')
    .config('spark.ui.showConsoleProgress', 'false')  # tắt spam log
    .config('spark.driver.extraJavaOptions',
            '-Dlog4j.rootCategory=ERROR,console')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')   # ẩn WARN

print('✅ SparkSession khởi tạo thành công!')
print(f'   Version : {spark.version}')
print(f'   App     : {spark.sparkContext.appName}')
print(f'   Master  : {spark.sparkContext.master}')


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/07 14:49:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ SparkSession khởi tạo thành công!
   Version : 4.0.2
   App     : Instacart_BigData_Analysis
   Master  : local[*]


## 📂 PHẦN 2 — Đọc dữ liệu


In [7]:
from pyspark.sql.types import IntegerType, DoubleType
import pyspark.sql.functions as F

BASE = '/kaggle/input/datasets/yasserh/instacart-online-grocery-basket-analysis-dataset'

# ── Tùy chọn đọc CSV ─────────────────────────────────────────────────────
csv_options = {
    "header": "true",
    "inferSchema": "true",
    "quote": '"',       # dấu bao chuỗi
    "escape": '"',      # escape bằng chính dấu quote (chuẩn RFC 4180)
    "multiLine": "true" # xử lý trường hợp newline bên trong chuỗi
}

orders_df = (
    spark.read.options(**csv_options).csv(f'{BASE}/orders.csv')
    .withColumnRenamed('days_since_prior_order', 'days_since_prior')
)

order_products_prior_df = spark.read.options(**csv_options).csv(f'{BASE}/order_products__prior.csv')
order_products_train_df = spark.read.options(**csv_options).csv(f'{BASE}/order_products__train.csv')

# ── products.csv: dùng schema tường minh, KHÔNG inferSchema ──────────────
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

products_schema = StructType([
    StructField("product_id",   IntegerType(), True),
    StructField("product_name", StringType(),  True),
    StructField("aisle_id",     IntegerType(), True),
    StructField("department_id",IntegerType(), True),
])

products_df = spark.read.options(
    header="true",
    quote='"',
    escape='"',
    multiLine="true"
).schema(products_schema).csv(f'{BASE}/products.csv')

aisles_df       = spark.read.options(**csv_options).csv(f'{BASE}/aisles.csv')
departments_df  = spark.read.options(**csv_options).csv(f'{BASE}/departments.csv')

# ── Cast kiểu dữ liệu ────────────────────────────────────────────────────
orders_df = (
    orders_df
    .withColumn("days_since_prior",   F.col("days_since_prior").cast(DoubleType()))
    .withColumn("order_dow",          F.col("order_dow").cast(DoubleType()))
    .withColumn("order_hour_of_day",  F.col("order_hour_of_day").cast(DoubleType()))
)

aisles_df      = aisles_df.withColumn("aisle_id",      F.col("aisle_id").cast(IntegerType()))
departments_df = departments_df.withColumn("department_id", F.col("department_id").cast(IntegerType()))

# ── Gộp prior + train ─────────────────────────────────────────────────────
order_products_all_df = order_products_prior_df.union(order_products_train_df)

# ── Đăng ký TempViews ─────────────────────────────────────────────────────
orders_df.createOrReplaceTempView("orders")
order_products_prior_df.createOrReplaceTempView("order_products_prior")
order_products_train_df.createOrReplaceTempView("order_products_train")
order_products_all_df.createOrReplaceTempView("order_products_all")
products_df.createOrReplaceTempView("products")
aisles_df.createOrReplaceTempView("aisles")
departments_df.createOrReplaceTempView("departments")

print('✅ Đọc xong và đăng ký TempViews!')

# Kiểm tra nhanh
products_df.show(3, truncate=False)
print("products schema:"); products_df.printSchema()

✅ Đọc xong và đăng ký TempViews!
+----------+------------------------------------+--------+-------------+
|product_id|product_name                        |aisle_id|department_id|
+----------+------------------------------------+--------+-------------+
|1         |Chocolate Sandwich Cookies          |61      |19           |
|2         |All-Seasons Salt                    |104     |13           |
|3         |Robust Golden Unsweetened Oolong Tea|94      |7            |
+----------+------------------------------------+--------+-------------+
only showing top 3 rows
products schema:
root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- aisle_id: integer (nullable = true)
 |-- department_id: integer (nullable = true)



## 📊 PHẦN 3 — Thông tin cơ bản (Shape / Size)


In [6]:
tables = {
    'orders':                orders_df,
    'order_products_prior':  order_products_prior_df,
    'order_products_train':  order_products_train_df,
    'order_products_all':    order_products_all_df,
    'products':              products_df,
    'aisles':                aisles_df,
    'departments':           departments_df,
}

print(f'{'Bảng':<30} {'Rows':>12} {'Cols':>6}')
print('-'*50)
for name, df in tables.items():
    r = df.count(); c = len(df.columns)
    print(f'{name:<30} {r:>12,} {c:>6}')

# In schema chi tiết từng bảng
for name, df in tables.items():
    print(f'\n── Schema: {name} ──')
    df.printSchema()


Bảng                                   Rows   Cols
--------------------------------------------------
orders                            3,421,083      7
order_products_prior             32,434,489      4
order_products_train              1,384,617      4
order_products_all               33,819,106      4
products                             49,688      4
aisles                                  134      2
departments                              21      2

── Schema: orders ──
root
 |-- order_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- eval_set: string (nullable = true)
 |-- order_number: integer (nullable = true)
 |-- order_dow: double (nullable = true)
 |-- order_hour_of_day: double (nullable = true)
 |-- days_since_prior: double (nullable = true)


── Schema: order_products_prior ──
root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- add_to_cart_order: integer (nullable = true)
 |-- reordered: integer (nullable 

## 🗄️ PHẦN 4 — Tạo TempViews


In [8]:
orders_df.createOrReplaceTempView('orders')
order_products_prior_df.createOrReplaceTempView('order_products_prior')
order_products_train_df.createOrReplaceTempView('order_products_train')
order_products_all_df.createOrReplaceTempView('order_products_all')
products_df.createOrReplaceTempView('products')
aisles_df.createOrReplaceTempView('aisles')
departments_df.createOrReplaceTempView('departments')

print('✅ TempViews đã tạo:')
for v in spark.catalog.listTables():
    print(f'   → {v.name}')


✅ TempViews đã tạo:
   → aisles
   → departments
   → order_products_all
   → order_products_prior
   → order_products_train
   → orders
   → products


## 🔍 PHẦN 5 — 20 Câu Spark SQL Queries Nâng Cao

Mỗi query được:
- Lưu vào **cache()** để tái sử dụng nhanh
- Xuất ra **Pandas DataFrame** để hiển thị
- Áp dụng các kỹ thuật: **GROUP BY + Aggregation**, **Window Functions**, **Subquery**, **JOIN nhiều bảng**, **CASE WHEN**, **CTE (WITH clause)**, **PERCENTILE_APPROX**


### 🔎 Q1: Phân bố đơn hàng theo giờ × ngày trong tuần (Heatmap data)
**Kỹ thuật:** `GROUP BY + CASE WHEN + Window (SUM OVER)`

**Insight:** xem kết quả bên dưới


In [8]:
q1_df = spark.sql("""
    SELECT
        order_dow,
        CASE order_dow
            WHEN 0 THEN 'Sunday'    WHEN 1 THEN 'Monday'
            WHEN 2 THEN 'Tuesday'   WHEN 3 THEN 'Wednesday'
            WHEN 4 THEN 'Thursday'  WHEN 5 THEN 'Friday'
            WHEN 6 THEN 'Saturday'
        END AS day_name,
        order_hour_of_day,
        COUNT(*)                                           AS total_orders,
        ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER(), 2)     AS pct_of_total
    FROM orders
    GROUP BY order_dow, day_name, order_hour_of_day
    ORDER BY order_dow, order_hour_of_day
""").cache()
q1_df.toPandas()


,order_dow,day_name,order_hour_of_day,total_orders,pct_of_total
0,0.0,Sunday,0.0,3936,0.12
1,0.0,Sunday,1.0,2398,0.07
2,0.0,Sunday,2.0,1409,0.04
3,0.0,Sunday,3.0,963,0.03
4,0.0,Sunday,4.0,813,0.02
...,...,...,...,...,...
163,6.0,Saturday,19.0,18346,0.54
164,6.0,Saturday,20.0,13392,0.39
165,6.0,Saturday,21.0,10501,0.31
166,6.0,Saturday,22.0,8532,0.25


### 🔎 Q2: Top 20 sản phẩm bán chạy nhất + reorder rate
**Kỹ thuật:** `JOIN 4 bảng + DENSE_RANK() Window`

**Insight:** xem kết quả bên dưới


In [9]:
q2_df = spark.sql("""
    SELECT
        p.product_id,
        p.product_name,
        a.aisle,
        d.department,
        COUNT(op.order_id)                                          AS total_sold,
        ROUND(SUM(op.reordered)*100.0/COUNT(op.order_id), 2)       AS reorder_rate_pct,
        DENSE_RANK() OVER (ORDER BY COUNT(op.order_id) DESC)        AS sales_rank
    FROM order_products_all op
    JOIN products    p ON op.product_id    = p.product_id
    JOIN aisles      a ON p.aisle_id       = a.aisle_id
    JOIN departments d ON p.department_id  = d.department_id
    GROUP BY p.product_id, p.product_name, a.aisle, d.department
    ORDER BY total_sold DESC
    LIMIT 20
""").cache()
q2_df.toPandas()


,product_id,product_name,aisle,department,total_sold,reorder_rate_pct,sales_rank
0,24852,Banana,fresh fruits,produce,491291,84.51,1
1,13176,Bag of Organic Bananas,fresh fruits,produce,394930,83.38,2
2,21137,Organic Strawberries,fresh fruits,produce,275577,77.82,3
3,21903,Organic Baby Spinach,packaged vegetables fruits,produce,251705,77.45,4
4,47209,Organic Hass Avocado,fresh fruits,produce,220877,79.76,5
5,47766,Organic Avocado,fresh fruits,produce,184224,76.14,6
6,47626,Large Lemon,fresh fruits,produce,160792,69.77,7
7,16797,Strawberries,fresh fruits,produce,149445,69.98,8
8,26209,Limes,fresh fruits,produce,146660,68.19,9
9,27845,Organic Whole Milk,milk,dairy eggs,142813,83.10,10


### 🔎 Q3: Tỷ lệ reorder trung bình theo department + rank
**Kỹ thuật:** `JOIN + GROUP BY + RANK() Window`

**Insight:** xem kết quả bên dưới


In [10]:
q3_df = spark.sql("""
    SELECT
        d.department,
        COUNT(op.order_id)                                             AS total_items,
        SUM(op.reordered)                                              AS total_reordered,
        ROUND(SUM(op.reordered)*100.0/COUNT(op.order_id), 2)          AS reorder_rate_pct,
        RANK() OVER (ORDER BY SUM(op.reordered)*1.0/COUNT(op.order_id) DESC) AS rnk
    FROM order_products_all op
    JOIN products    p ON op.product_id    = p.product_id
    JOIN departments d ON p.department_id  = d.department_id
    GROUP BY d.department
    ORDER BY reorder_rate_pct DESC
""").cache()
q3_df.toPandas()


,department,total_items,total_reordered,reorder_rate_pct,rnk
0,dairy eggs,5631067,3773723,67.02,1
1,beverages,2804175,1832952,65.37,2
2,produce,9888378,6432596,65.05,3
3,bakery,1225181,769880,62.84,4
4,deli,1095540,666231,60.81,5
5,pets,102221,61594,60.26,6
6,babies,438743,253453,57.77,7
7,bulk,35932,20736,57.71,8
8,snacks,3006412,1727075,57.45,9
9,alcohol,159294,90992,57.12,10


### 🔎 Q4: Phân khúc khách hàng theo tần suất mua hàng (RFM-style)
**Kỹ thuật:** `CTE + CASE WHEN + Percent OVER (Window)`

**Insight:** xem kết quả bên dưới


In [11]:
q4_df = spark.sql("""
    WITH user_stats AS (
        SELECT
            user_id,
            COUNT(DISTINCT order_id)        AS total_orders,
            AVG(days_since_prior)           AS avg_days_between_orders,
            MAX(order_number)               AS max_order_number
        FROM orders
        GROUP BY user_id
    )
    SELECT
        CASE
            WHEN total_orders >= 20 THEN 'Champion (≥20 đơn)'
            WHEN total_orders >= 10 THEN 'Loyal (10-19 đơn)'
            WHEN total_orders >= 5  THEN 'Regular (5-9 đơn)'
            ELSE                         'New (1-4 đơn)'
        END AS customer_segment,
        COUNT(*)                                         AS num_customers,
        ROUND(AVG(total_orders), 1)                      AS avg_orders,
        ROUND(AVG(avg_days_between_orders), 1)           AS avg_days_between,
        ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER(), 2)   AS pct_customers
    FROM user_stats
    GROUP BY customer_segment
    ORDER BY avg_orders DESC
""").cache()
q4_df.toPandas()


,customer_segment,num_customers,avg_orders,avg_days_between,pct_customers
0,Champion (≥20 đơn),53931,38.4,9.2,26.15
1,Loyal (10-19 đơn),56797,13.6,15.3,27.54
2,Regular (5-9 đơn),71495,6.7,18.7,34.67
3,New (1-4 đơn),23986,4.0,20.3,11.63


### 🔎 Q5: Kích thước giỏ hàng trung bình theo từng giờ trong ngày
**Kỹ thuật:** `JOIN + GROUP BY + AVG Aggregation`

**Insight:** xem kết quả bên dưới


In [12]:
q5_df = spark.sql("""
    SELECT
        o.order_hour_of_day,
        COUNT(DISTINCT o.order_id)                         AS num_orders,
        COUNT(op.product_id)                               AS total_items,
        ROUND(COUNT(op.product_id)*1.0/COUNT(DISTINCT o.order_id), 2) AS avg_basket_size,
        ROUND(AVG(op.reordered)*100, 2)                    AS avg_reorder_pct
    FROM orders o
    JOIN order_products_all op ON o.order_id = op.order_id
    GROUP BY o.order_hour_of_day
    ORDER BY o.order_hour_of_day
""").cache()
q5_df.toPandas()


,order_hour_of_day,num_orders,total_items,avg_basket_size,avg_reorder_pct
0,0.0,22224,228031,10.26,56.57
1,1.0,12103,121412,10.03,55.81
2,2.0,7375,72660,9.85,55.56
3,3.0,5343,53759,10.06,56.05
4,4.0,5393,55714,10.33,57.24
5,5.0,9374,91909,9.80,60.86
6,6.0,29913,302642,10.12,63.70
7,7.0,90032,928239,10.31,64.47
8,8.0,174664,1787359,10.23,63.23
9,9.0,252529,2550569,10.10,61.96


### 🔎 Q6: Phân bố khoảng cách giữa các lần mua – chuỗi thời gian
**Kỹ thuật:** `Window: cumulative SUM (Running Total) – Chuỗi thời gian`

**Insight:** xem kết quả bên dưới


In [13]:
q6_df = spark.sql("""
    SELECT
        days_since_prior,
        COUNT(*)                                              AS num_orders,
        ROUND(COUNT(*)*100.0/SUM(COUNT(*)) OVER(), 3)        AS pct,
        SUM(COUNT(*)) OVER (
            ORDER BY days_since_prior
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )                                                     AS cumulative_orders
    FROM orders
    WHERE days_since_prior IS NOT NULL
    GROUP BY days_since_prior
    ORDER BY days_since_prior
""").cache()
q6_df.toPandas()


,days_since_prior,num_orders,pct,cumulative_orders
0,0.0,67755,2.108,67755
1,1.0,145247,4.518,213002
2,2.0,193206,6.010,406208
3,3.0,217005,6.750,623213
4,4.0,221696,6.896,844909
5,5.0,214503,6.672,1059412
6,6.0,240013,7.466,1299425
7,7.0,320608,9.973,1620033
8,8.0,181717,5.652,1801750
9,9.0,118188,3.676,1919938


### 🔎 Q7: Top 15 aisle theo doanh số + dual ranking (sales vs reorder)
**Kỹ thuật:** `JOIN 4 bảng + 2 RANK() Windows khác nhau`

**Insight:** xem kết quả bên dưới


In [14]:
q7_df = spark.sql("""
    SELECT
        a.aisle,
        d.department,
        COUNT(op.order_id)                              AS total_sold,
        COUNT(DISTINCT p.product_id)                   AS unique_products,
        ROUND(AVG(op.reordered)*100, 2)                AS reorder_rate_pct,
        RANK() OVER (ORDER BY COUNT(op.order_id) DESC) AS sales_rank,
        RANK() OVER (ORDER BY AVG(op.reordered) DESC)  AS reorder_rank
    FROM order_products_all op
    JOIN products    p ON op.product_id    = p.product_id
    JOIN aisles      a ON p.aisle_id       = a.aisle_id
    JOIN departments d ON p.department_id  = d.department_id
    GROUP BY a.aisle, d.department
    ORDER BY total_sold DESC
    LIMIT 15
""").cache()
q7_df.toPandas()


,aisle,department,total_sold,unique_products,reorder_rate_pct,sales_rank,reorder_rank
0,fresh fruits,produce,3792661,382,71.88,1,3
1,fresh vegetables,produce,3568630,569,59.51,2,30
2,packaged vegetables fruits,produce,1843806,615,63.93,3,14
3,yogurt,dairy eggs,1507583,1026,68.65,4,7
4,packaged cheese,dairy eggs,1021462,891,58.57,5,37
5,milk,dairy eggs,923659,243,78.18,6,1
6,water seltzer sparkling water,beverages,878150,344,72.99,7,2
7,chips pretzels,snacks,753739,989,58.91,8,34
8,soy lactosefree,dairy eggs,664493,293,69.24,9,5
9,bread,bakery,608469,557,67.06,10,9


### 🔎 Q8: Organic vs Non-Organic – So sánh hành vi mua hàng
**Kỹ thuật:** `JOIN + LOWER() string + CASE WHEN + GROUP BY`

**Insight:** xem kết quả bên dưới


In [15]:
q8_df = spark.sql("""
    SELECT
        CASE WHEN LOWER(p.product_name) LIKE '%organic%' THEN 'Organic'
             ELSE 'Non-Organic' END                AS product_type,
        COUNT(op.order_id)                           AS total_items_sold,
        COUNT(DISTINCT p.product_id)                 AS unique_products,
        ROUND(AVG(op.reordered)*100, 2)              AS avg_reorder_rate,
        ROUND(AVG(op.add_to_cart_order), 2)          AS avg_cart_position
    FROM order_products_all op
    JOIN products p ON op.product_id = p.product_id
    GROUP BY product_type
    ORDER BY total_items_sold DESC
""").cache()
q8_df.toPandas()


,product_type,total_items_sold,unique_products,avg_reorder_rate,avg_cart_position
0,Non-Organic,23162774,44649,56.92,8.35
1,Organic,10656332,5036,63.54,8.41


### 🔎 Q9: Running total đơn hàng theo order sequence (loyalty progression)
**Kỹ thuật:** `Window: SUM OVER (ORDER BY) – Cumulative percentage`

**Insight:** xem kết quả bên dưới


In [16]:
q9_df = spark.sql("""
    WITH order_seq AS (
        SELECT order_number, COUNT(*) AS orders_at_seq
        FROM orders
        GROUP BY order_number
    )
    SELECT
        order_number,
        orders_at_seq,
        SUM(orders_at_seq) OVER (ORDER BY order_number)  AS cumulative_orders,
        ROUND(SUM(orders_at_seq) OVER (ORDER BY order_number)
              *100.0/SUM(orders_at_seq) OVER (), 2)      AS cumulative_pct,
        ROUND(orders_at_seq*100.0/SUM(orders_at_seq) OVER (), 3) AS pct_at_seq
    FROM order_seq
    ORDER BY order_number
""").cache()
q9_df.toPandas().head(25)


,order_number,orders_at_seq,cumulative_orders,cumulative_pct,pct_at_seq
0,1,206209,206209,6.03,6.028
1,2,206209,412418,12.06,6.028
2,3,206209,618627,18.08,6.028
3,4,206209,824836,24.11,6.028
4,5,182223,1007059,29.44,5.326
5,6,162633,1169692,34.19,4.754
6,7,146468,1316160,38.47,4.281
7,8,132618,1448778,42.35,3.876
8,9,120918,1569696,45.88,3.534
9,10,110728,1680424,49.12,3.237


### 🔎 Q10: Hidden Gems – sản phẩm reorder cao nhưng volume thấp (Subquery)
**Kỹ thuật:** `Subquery (Inline View) + JOIN 4 bảng + HAVING`

**Insight:** xem kết quả bên dưới


In [17]:
q10_df = spark.sql("""
    SELECT *
    FROM (
        SELECT
            p.product_name,
            a.aisle,
            d.department,
            COUNT(op.order_id)              AS total_orders,
            ROUND(AVG(op.reordered)*100, 2) AS reorder_rate_pct,
            COUNT(DISTINCT o.user_id)       AS unique_buyers
        FROM order_products_all op
        JOIN products    p ON op.product_id    = p.product_id
        JOIN aisles      a ON p.aisle_id       = a.aisle_id
        JOIN departments d ON p.department_id  = d.department_id
        JOIN orders      o ON op.order_id      = o.order_id
        GROUP BY p.product_name, a.aisle, d.department
    ) ranked
    WHERE reorder_rate_pct > 70
      AND total_orders BETWEEN 200 AND 2000
    ORDER BY reorder_rate_pct DESC, total_orders ASC
    LIMIT 20
""").cache()
q10_df.toPandas()


,product_name,aisle,department,total_orders,reorder_rate_pct,unique_buyers
0,Real2 Alkalized Water 500 ml,water seltzer sparkling water,beverages,457,86.21,63
1,Lo-Carb Energy Drink,energy sports drinks,beverages,484,85.95,68
2,Ultra-Purified Water,water seltzer sparkling water,beverages,1524,85.70,218
3,Homestyle Orange Juice,refrigerated,beverages,317,85.17,47
4,Raw Whole Milk,milk,dairy eggs,241,84.65,37
5,"Purified Water, 9.5pH+",water seltzer sparkling water,beverages,319,84.33,50
6,Lowfat Goat Milk,milk,dairy eggs,1198,83.81,194
7,Alkalized Water,water seltzer sparkling water,beverages,610,83.61,100
8,Purified Alkalkine Water with Minerals pH10,water seltzer sparkling water,beverages,749,83.44,124
9,Organic Grade A Raw Whole Milk,milk,dairy eggs,267,83.15,45


### 🔎 Q11: First-time buy vs Reorder phân tích theo department
**Kỹ thuật:** `JOIN + CASE WHEN inside SUM + GROUP BY`

**Insight:** xem kết quả bên dưới


In [18]:
q11_df = spark.sql("""
    SELECT
        d.department,
        SUM(CASE WHEN op.reordered = 0 THEN 1 ELSE 0 END) AS first_time_buys,
        SUM(CASE WHEN op.reordered = 1 THEN 1 ELSE 0 END) AS reorders,
        COUNT(*)                                           AS total,
        ROUND(SUM(CASE WHEN op.reordered=0 THEN 1 ELSE 0 END)*100.0/COUNT(*),2) AS first_buy_pct,
        ROUND(SUM(CASE WHEN op.reordered=1 THEN 1 ELSE 0 END)*100.0/COUNT(*),2) AS reorder_pct
    FROM order_products_all op
    JOIN products    p ON op.product_id    = p.product_id
    JOIN departments d ON p.department_id  = d.department_id
    GROUP BY d.department
    ORDER BY reorder_pct DESC
""").cache()
q11_df.toPandas()

,department,first_time_buys,reorders,total,first_buy_pct,reorder_pct
0,dairy eggs,1857344,3773723,5631067,32.98,67.02
1,beverages,971223,1832952,2804175,34.63,65.37
2,produce,3455782,6432596,9888378,34.95,65.05
3,bakery,455301,769880,1225181,37.16,62.84
4,deli,429309,666231,1095540,39.19,60.81
5,pets,40627,61594,102221,39.74,60.26
6,babies,185290,253453,438743,42.23,57.77
7,bulk,15196,20736,35932,42.29,57.71
8,snacks,1279337,1727075,3006412,42.55,57.45
9,alcohol,68302,90992,159294,42.88,57.12


### 🔎 Q12: Top cặp sản phẩm hay được mua cùng nhau (Co-occurrence / Market Basket)
**Kỹ thuật:** `Self-JOIN + CTE + IN Subquery (Popular products filter)`

**Insight:** xem kết quả bên dưới


In [19]:
q12_df = spark.sql("""
    WITH popular AS (
        SELECT product_id
        FROM order_products_prior
        GROUP BY product_id
        HAVING COUNT(*) > 50000
    ),
    pairs AS (
        SELECT
            a.order_id,
            a.product_id AS prod_a,
            b.product_id AS prod_b
        FROM order_products_prior a
        JOIN order_products_prior b
             ON a.order_id = b.order_id AND a.product_id < b.product_id
        WHERE a.product_id IN (SELECT product_id FROM popular)
          AND b.product_id IN (SELECT product_id FROM popular)
    )
    SELECT
        p1.product_name AS product_a,
        p2.product_name AS product_b,
        COUNT(*)         AS co_occurrence
    FROM pairs
    JOIN products p1 ON pairs.prod_a = p1.product_id
    JOIN products p2 ON pairs.prod_b = p2.product_id
    GROUP BY p1.product_name, p2.product_name
    ORDER BY co_occurrence DESC
    LIMIT 20
""").cache()
q12_df.toPandas()


,product_a,product_b,co_occurrence
0,Bag of Organic Bananas,Organic Hass Avocado,62341
1,Bag of Organic Bananas,Organic Strawberries,61628
2,Organic Strawberries,Banana,56156
3,Banana,Organic Avocado,53395
4,Organic Baby Spinach,Banana,51395
5,Bag of Organic Bananas,Organic Baby Spinach,50372
6,Strawberries,Banana,41232
7,Banana,Large Lemon,40880
8,Organic Strawberries,Organic Hass Avocado,40794
9,Bag of Organic Bananas,Organic Raspberries,40503


### 🔎 Q13: Loyalty tier analysis – basket size & reorder rate theo độ trung thành
**Kỹ thuật:** `CTE + JOIN + CASE WHEN phân tầng + nested subquery`

**Insight:** xem kết quả bên dưới


In [20]:
q13_df = spark.sql("""
    WITH user_loyalty AS (
        SELECT user_id, MAX(order_number) AS loyalty_score
        FROM orders WHERE eval_set = 'prior'
        GROUP BY user_id
    ),
    user_basket AS (
        SELECT
            o.user_id,
            AVG(bk.cnt)           AS avg_basket_size,
            AVG(op.reordered)     AS avg_reorder_rate
        FROM orders o
        JOIN order_products_prior op ON o.order_id = op.order_id
        JOIN (
            SELECT order_id, COUNT(*) AS cnt
            FROM order_products_prior GROUP BY order_id
        ) bk ON o.order_id = bk.order_id
        GROUP BY o.user_id
    )
    SELECT
        CASE
            WHEN ul.loyalty_score >= 50 THEN 'Super Loyal (≥50)'
            WHEN ul.loyalty_score >= 20 THEN 'Loyal (20-49)'
            WHEN ul.loyalty_score >= 10 THEN 'Regular (10-19)'
            ELSE 'Casual (<10)'
        END AS loyalty_tier,
        COUNT(*)                              AS num_users,
        ROUND(AVG(ul.loyalty_score), 1)       AS avg_orders,
        ROUND(AVG(ub.avg_basket_size), 2)     AS avg_basket_size,
        ROUND(AVG(ub.avg_reorder_rate)*100,2) AS avg_reorder_rate_pct
    FROM user_loyalty ul
    JOIN user_basket ub ON ul.user_id = ub.user_id
    GROUP BY loyalty_tier
    ORDER BY avg_orders DESC
""").cache()
q13_df.toPandas()
# Chia theo phân vị

,loyalty_tier,num_users,avg_orders,avg_basket_size,avg_reorder_rate_pct
0,Super Loyal (≥50),10910,68.9,12.49,73.63
1,Loyal (20-49),39821,30.3,12.59,62.57
2,Regular (10-19),50965,13.7,12.09,48.83
3,Casual (<10),104513,5.3,11.37,29.95


### 🔎 Q14: Department nào bán chạy nhất trong từng khung giờ (Cross-dimension)
**Kỹ thuật:** `CTE + CASE WHEN time bucket + PARTITION BY Window RANK`

**Insight:** xem kết quả bên dưới


In [51]:
q14_df = spark.sql("""
    WITH dept_hour AS (
        SELECT
            CASE
                WHEN o.order_hour_of_day BETWEEN 6  AND 11 THEN '06-11 Sáng'
                WHEN o.order_hour_of_day BETWEEN 12 AND 17 THEN '12-17 Chiều'
                WHEN o.order_hour_of_day BETWEEN 18 AND 21 THEN '18-21 Tối'
                ELSE '22-05 Đêm/Khuya'
            END AS time_slot,
            d.department,
            COUNT(*) AS items_sold
        FROM orders o
        JOIN order_products_all op ON o.order_id    = op.order_id
        JOIN products           p  ON op.product_id = p.product_id
        JOIN departments        d  ON p.department_id = d.department_id
        GROUP BY time_slot, d.department
    ),
    ranked AS (
        SELECT *,
            RANK() OVER (PARTITION BY time_slot ORDER BY items_sold DESC) AS rank
        FROM dept_hour
    )
    SELECT time_slot, department, items_sold, rank
    FROM ranked
    WHERE rank <= 5
    ORDER BY time_slot, rank
""").cache()
q14_df.toPandas()


,time_slot,department,items_sold,rank
0,06-11 Sáng,produce,3305825,1
1,06-11 Sáng,dairy eggs,1941826,2
2,06-11 Sáng,snacks,1035165,3
3,06-11 Sáng,beverages,967649,4
4,06-11 Sáng,frozen,704040,5
5,12-17 Chiều,produce,4638072,1
6,12-17 Chiều,dairy eggs,2603925,2
7,12-17 Chiều,snacks,1416890,3
8,12-17 Chiều,beverages,1313995,4
9,12-17 Chiều,frozen,1133152,5


### 🔎 Q15: Impulse products – sản phẩm thường được thêm vào giỏ hàng sớm nhất
**Kỹ thuật:** `JOIN 4 bảng + AVG(add_to_cart_order) + HAVING filter`

**Insight:** xem kết quả bên dưới


In [22]:
q15_df = spark.sql("""
    SELECT
        p.product_name,
        a.aisle,
        d.department,
        ROUND(AVG(op.add_to_cart_order), 3) AS avg_cart_position,
        COUNT(*)                             AS frequency,
        ROUND(AVG(op.reordered)*100, 2)      AS reorder_rate_pct
    FROM order_products_all op
    JOIN products    p ON op.product_id    = p.product_id
    JOIN aisles      a ON p.aisle_id       = a.aisle_id
    JOIN departments d ON p.department_id  = d.department_id
    GROUP BY p.product_name, a.aisle, d.department
    HAVING COUNT(*) > 10000
    ORDER BY avg_cart_position ASC
    LIMIT 20
""").cache()
q15_df.toPandas()


,product_name,aisle,department,avg_cart_position,frequency,reorder_rate_pct
0,Extra Fancy Unsalted Mixed Nuts,nuts seeds dried fruit,snacks,3.281,10030,81.84
1,Baby Cucumbers,packaged produce,produce,3.460,15031,78.51
2,0% Greek Strained Yogurt,yogurt,dairy eggs,3.494,13651,82.68
3,Sweet Kale Salad Mix,packaged produce,produce,3.597,11953,71.29
4,Soda,soft drinks,beverages,3.732,37298,77.78
5,Apples,fresh fruits,produce,3.768,12883,77.82
6,Trail Mix,trail mix snack mix,snacks,3.802,12601,78.26
7,Packaged Grape Tomatoes,packaged produce,produce,3.985,13965,76.73
8,Clementines,packaged produce,produce,4.100,32194,75.90
9,Smartwater,water seltzer sparkling water,beverages,4.106,11874,73.67


### 🔎 Q16: Phân tích đơn hàng cuối cùng (train set) – department breakdown
**Kỹ thuật:** `JOIN train table riêng + GROUP BY department`

**Insight:** xem kết quả bên dưới


In [23]:
q16_df = spark.sql("""
    SELECT
        d.department,
        COUNT(op.order_id)                  AS items_in_train,
        COUNT(DISTINCT op.order_id)          AS orders_in_train,
        ROUND(AVG(op.reordered)*100, 2)      AS reorder_rate_pct,
        ROUND(AVG(op.add_to_cart_order), 2)  AS avg_cart_position
    FROM order_products_train op
    JOIN products    p ON op.product_id    = p.product_id
    JOIN departments d ON p.department_id  = d.department_id
    GROUP BY d.department
    ORDER BY items_in_train DESC
""").cache()
q16_df.toPandas()


,department,items_in_train,orders_in_train,reorder_rate_pct,avg_cart_position
0,produce,409087,96927,66.46,8.43
1,dairy eggs,217051,87400,67.50,7.88
2,snacks,118862,57302,58.14,9.56
3,beverages,114046,61482,65.82,7.14
4,frozen,100426,51071,55.93,9.44
5,pantry,81242,47599,36.31,10.12
6,bakery,48394,36424,63.42,8.54
7,canned goods,46799,29416,48.68,10.36
8,deli,44291,32281,61.79,9.05
9,dry goods pasta,38713,25876,48.78,10.80


### 🔎 Q17: Top 10 user hoạt động nhất – đặc điểm mua sắm chi tiết
**Kỹ thuật:** `CTE + JOIN 3 bảng + GROUP BY user`

**Insight:** xem kết quả bên dưới


In [24]:
q17_df = spark.sql("""
    WITH top_users AS (
        SELECT user_id, COUNT(DISTINCT order_id) AS total_orders
        FROM orders
        GROUP BY user_id
        ORDER BY total_orders DESC
        LIMIT 10
    )
    SELECT
        tu.user_id,
        tu.total_orders,
        COUNT(DISTINCT op.product_id)         AS unique_products,
        SUM(op.reordered)                     AS total_reorders,
        ROUND(AVG(op.reordered)*100, 2)       AS reorder_rate_pct,
        ROUND(AVG(o.days_since_prior), 1)     AS avg_days_between_orders
    FROM top_users tu
    JOIN orders             o  ON tu.user_id   = o.user_id
    JOIN order_products_all op ON o.order_id   = op.order_id
    GROUP BY tu.user_id, tu.total_orders
    ORDER BY tu.total_orders DESC
""").cache()
q17_df.toPandas()


,user_id,total_orders,unique_products,total_reorders,reorder_rate_pct,avg_days_between_orders
0,9667,100,253,493,66.09,4.3
1,12641,100,378,1093,74.30,4.2
2,4382,100,262,458,63.61,2.9
3,8086,100,73,421,85.22,3.6
4,10293,100,303,734,70.78,3.5
5,4807,100,227,402,63.91,3.2
6,8085,100,200,652,76.53,1.8
7,5329,100,294,347,54.13,2.1
8,4711,100,159,1018,86.49,3.1
9,12102,100,211,1068,83.50,4.0


### 🔎 Q18: Basket size & reorder rate thay đổi theo lần mua thứ N (Time Series)
**Kỹ thuật:** `JOIN + GROUP BY order_number + STDDEV() + chuỗi thời gian theo loyalty`

**Insight:** xem kết quả bên dưới


In [25]:
q18_df = spark.sql("""
    WITH basket_by_seq AS (
        SELECT
            o.order_number,
            o.order_id,
            COUNT(op.product_id) AS basket_size,
            AVG(op.reordered)    AS reorder_rate
        FROM orders o
        JOIN order_products_prior op ON o.order_id = op.order_id
        WHERE o.order_number <= 30
        GROUP BY o.order_number, o.order_id
    )
    SELECT
        order_number,
        COUNT(order_id)                     AS num_orders,
        ROUND(AVG(basket_size), 2)          AS avg_basket_size,
        ROUND(STDDEV(basket_size), 2)       AS stddev_basket_size,
        ROUND(MIN(basket_size), 0)          AS min_basket,
        ROUND(MAX(basket_size), 0)          AS max_basket,
        ROUND(AVG(reorder_rate)*100, 2)     AS avg_reorder_pct
    FROM basket_by_seq
    GROUP BY order_number
    ORDER BY order_number
""").cache()
q18_df.toPandas()

# Tại đơn hàng thứ 15 (order_number = 15), nếu avg_reorder_pct = 70.93%, 
# điều này có nghĩa là: Trung bình một giỏ hàng ở lượt mua thứ 15 của khách hàng sẽ có 70.93% sản phẩm là đồ cũ họ đã từng mua trước đây  
# và chỉ có 28% là các sản phẩm mới hoàn toàn

,order_number,num_orders,avg_basket_size,stddev_basket_size,min_basket,max_basket,avg_reorder_pct
0,1,206209,10.08,7.47,1,98,0.00
1,2,206209,9.93,7.40,1,108,28.84
2,3,206209,9.94,7.47,1,127,40.51
3,4,182223,9.99,7.50,1,145,47.71
4,5,162633,10.01,7.49,1,92,52.59
5,6,146468,10.05,7.52,1,96,56.18
6,7,132618,10.06,7.53,1,104,59.16
7,8,120918,10.08,7.54,1,105,61.37
8,9,110728,10.12,7.55,1,94,63.40
9,10,101696,10.12,7.55,1,98,64.99


### 🔎 Q19: Funnel phân tích vị trí giỏ hàng – reorder rate theo add_to_cart position
**Kỹ thuật:** `CASE WHEN bucketing + GROUP BY + AVG Aggregation`

**Insight:** xem kết quả bên dưới


In [26]:
q19_df = spark.sql("""
    SELECT
        CASE
            WHEN add_to_cart_order = 1   THEN '1 - First item'
            WHEN add_to_cart_order <= 3  THEN '2-3 Early'
            WHEN add_to_cart_order <= 7  THEN '4-7 Middle-early'
            WHEN add_to_cart_order <= 15 THEN '8-15 Middle'
            ELSE '16+ Late'
        END AS cart_position_bucket,
        COUNT(*)                         AS total_items,
        ROUND(AVG(reordered)*100, 2)     AS avg_reorder_rate_pct,
        ROUND(AVG(add_to_cart_order), 1) AS avg_position
    FROM order_products_all
    GROUP BY cart_position_bucket
    ORDER BY avg_position
""").cache()
q19_df.toPandas()
# Nghiên cứu thử phương pháp thống kê binning

,cart_position_bucket,total_items,avg_reorder_rate_pct,avg_position
0,1 - First item,3346083,67.93,1.0
1,2-3 Early,6170619,66.81,2.5
2,4-7 Middle-early,9690546,61.24,5.4
3,8-15 Middle,9859148,54.51,10.8
4,16+ Late,4752710,47.36,22.2


In [9]:
q19_equal_freq_df = spark.sql("""
    WITH stats AS (
        -- Bước 1: Tính tổng số dòng và các điểm phân vị mốc 20%, 40%, 60%, 80%
        SELECT
            COUNT(*) AS total_count,
            percentile_approx(add_to_cart_order, 0.2) AS p20,
            percentile_approx(add_to_cart_order, 0.4) AS p40,
            percentile_approx(add_to_cart_order, 0.6) AS p60,
            percentile_approx(add_to_cart_order, 0.8) AS p80
        FROM order_products_all
    ),
    bucketed AS (
        -- Bước 2: Ánh xạ động từng dòng dữ liệu vào 5 thùng dựa trên ranh giới vừa tính
        SELECT
            op.add_to_cart_order,
            op.reordered,
            s.total_count,
            CASE
                WHEN op.add_to_cart_order <= s.p20 THEN CONCAT('1. Very Early (<=', CAST(s.p20 AS INT), ')')
                WHEN op.add_to_cart_order <= s.p40 THEN CONCAT('2. Early (<=', CAST(s.p40 AS INT), ')')
                WHEN op.add_to_cart_order <= s.p60 THEN CONCAT('3. Middle (<=', CAST(s.p60 AS INT), ')')
                WHEN op.add_to_cart_order <= s.p80 THEN CONCAT('4. Middle-Late (<=', CAST(s.p80 AS INT), ')')
                ELSE CONCAT('5. Late (>', CAST(s.p80 AS INT), ')')
            END AS cart_position_bucket
        FROM order_products_all op
        CROSS JOIN stats s -- Phép Join cực nhanh vì bảng stats chỉ có đúng 1 dòng
    )
    -- Bước 3: Gom nhóm và tính toán các chỉ số đầu ra
    SELECT
        cart_position_bucket,
        COUNT(*)                                        AS total_items,
        ROUND(COUNT(*) * 100.0 / FIRST(total_count), 2) AS pct_items_share, -- Xem tỷ lệ thực tế của mỗi thùng
        ROUND(AVG(reordered)*100, 2)                    AS avg_reorder_rate_pct,
        ROUND(AVG(add_to_cart_order), 1)                AS avg_position
    FROM bucketed
    GROUP BY cart_position_bucket
    ORDER BY avg_position
""").cache()

# Hiển thị kết quả kiểm tra
q19_equal_freq_df.toPandas()

,cart_position_bucket,total_items,pct_items_share,avg_reorder_rate_pct,avg_position
0,1. Very Early (<=3),9516702,28.14,67.20,2.0
1,2. Early (<=5),5315839,15.72,62.79,4.5
2,3. Middle (<=8),6215322,18.38,58.77,6.9
3,4. Middle-Late (<=13),6419772,18.98,54.49,10.7
4,5. Late (>13),6351471,18.78,48.36,20.2


### 🔎 Q20: Portfolio efficiency – phân bổ doanh số theo quy tắc Pareto 80/20
**Kỹ thuật:** `CTE + PERCENTILE_APPROX + SUM CASE WHEN + JOIN + GROUP BY`

**Insight:** xem kết quả bên dưới


In [27]:
q20_df = spark.sql("""
    WITH dept_product_stats AS (
        SELECT
            d.department,
            p.product_id,
            COUNT(op.order_id) AS total_sold,
            AVG(op.reordered)  AS reorder_rate
        FROM order_products_all op
        JOIN products    p ON op.product_id    = p.product_id
        JOIN departments d ON p.department_id  = d.department_id
        GROUP BY d.department, p.product_id
    ),
    -- Tính toán trước ngưỡng phân vị 80% (p80) cho từng phòng ban bằng Window Function
    with_percentile AS (
        SELECT
            *,
            percentile_approx(total_sold, 0.8) OVER (PARTITION BY department) AS dept_p80
        FROM dept_product_stats
    )
    -- Thực hiện gom nhóm chính
    SELECT
        department,
        COUNT(product_id)                        AS num_products,
        SUM(total_sold)                          AS total_items_sold,
        ROUND(AVG(total_sold), 1)                AS avg_sold_per_product,
        ROUND(AVG(reorder_rate)*100, 2)          AS avg_reorder_rate_pct,
        ROUND(
            SUM(CASE WHEN total_sold >= dept_p80 THEN total_sold ELSE 0 END)
            * 100.0 / NULLIF(SUM(total_sold), 0)
        , 2) AS top20pct_products_sales_share
    FROM with_percentile
    GROUP BY department
    ORDER BY total_items_sold DESC
""").cache()

q20_df.toPandas()

,department,num_products,total_items_sold,avg_sold_per_product,avg_reorder_rate_pct,top20pct_products_sales_share
0,produce,1684,9888378,5872.0,41.40,92.37
1,dairy eggs,3449,5631067,1632.7,50.70,87.03
2,snacks,6264,3006412,480.0,44.02,84.50
3,beverages,4364,2804175,642.6,47.27,87.62
4,frozen,4007,2336858,583.2,42.10,82.77
5,pantry,5370,1956819,364.4,24.46,86.22
6,bakery,1516,1225181,808.2,47.35,83.53
7,canned goods,2092,1114857,532.9,35.69,86.35
8,deli,1322,1095540,828.7,45.80,86.09
9,dry goods pasta,1858,905340,487.3,35.52,84.52


---
## 📊 PHẦN 6 — Dashboard Trực Quan Hóa

> Ba dashboard được tạo từ kết quả của các queries trên


In [28]:
# print('='*65)
# print(' HOÀN THÀNH PHÂN TÍCH BIG DATA – INSTACART!')
# print('   Outputs:')
# print('    dashboard_1_order_patterns.png')
# print('    dashboard_2_product_category.png')
# print('    dashboard_3_advanced_insights.png')
# print('='*65)
# spark.stop()
# print('SparkSession đã dừng.')


In [29]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

# Thu thập dữ liệu sang Pandas (giữ nguyên cấu trúc cũ của bạn)
q1_pd  = q1_df.toPandas()
q2_pd  = q2_df.toPandas()
q3_pd  = q3_df.toPandas()
q4_pd  = q4_df.toPandas()
q5_pd  = q5_df.toPandas()
q6_pd  = q6_df.toPandas()
q7_pd  = q7_df.toPandas()
q8_pd  = q8_df.toPandas()
q10_pd = q10_df.toPandas()
q11_pd = q11_df.toPandas()
q12_pd = q12_df.toPandas()
q13_pd = q13_df.toPandas()
q14_pd = q14_df.toPandas()
q18_pd = q18_df.toPandas()
q19_pd = q19_df.toPandas()
q20_pd = q20_df.toPandas()

# Cấu hình bảng màu hiện đại (Hex code giống hệt phiên bản cũ)
PALETTE_SEQ    = '#3B82F6' # Blue
PALETTE_ACCENT = '#F59E0B' # Orange
PALETTE_GREEN  = '#10B981' # Green
PALETTE_RED    = '#EF4444' # Red

print('✅ Data collected and Plotly libraries initialized!')

✅ Data collected and Plotly libraries initialized!


### **01. INSTACART – ORDER PATTERNS CHART**

In [30]:
import plotly.graph_objects as go

# 1. Chuẩn bị và biến đổi dữ liệu
pivot_1a = q1_pd.pivot(index='day_name', columns='order_hour_of_day', values='total_orders')
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday', 'Sunday']
pivot_1a = pivot_1a.reindex([d for d in day_order if d in pivot_1a.index])

# 2. Tạo biểu đồ độc lập
fig_1a = go.Figure(data=go.Heatmap(
    z=pivot_1a.values,
    x=pivot_1a.columns,
    y=pivot_1a.index,
    colorscale='YlGnBu',  # Bảng màu YlGnBu theo yêu cầu
    showscale=True,       # Hiển thị thanh nhiệt (heatbar)
    colorbar=dict(
        title=dict(text='<b>Orders</b>', font=dict(size=10)),
        thickness=15,
        tickfont=dict(size=9)
    ),
    hovertemplate='<b>Day:</b> %{y}<br><b>Hour:</b> %{x}h<br><b>Orders:</b> %{z:,.0f}<extra></extra>'
))

# 3. Định cấu hình trục và giao diện
fig_1a.update_layout(
    title='<b>Frequency of Day of week Vs Hour of day </b>',
    title_font=dict(size=15, family='Arial'),
    xaxis=dict(
        title='Hour of Day',
        tickmode='linear',
        tick0=0,
        dtick=1
    ),
    yaxis=dict(title='Day of Week'),
    template='plotly_white',
    height=500,
    width=950
)

fig_1a.show()

In [31]:
import plotly.graph_objects as go

# Tạo biểu đồ Bar
fig_1b = go.Figure(data=go.Bar(
    x=q5_pd['order_hour_of_day'],
    y=q5_pd['avg_basket_size'],
    marker_color='#3B82F6',  # PALETTE_SEQ
    hovertemplate='<b>Hour:</b> %{x}h<br><b>Avg Basket Size:</b> %{y:.2f} items<extra></extra>'
))

# Định cấu hình trục và giao diện
fig_1b.update_layout(
    title='<b>Avg Basket Size by Hour</b>',
    title_font=dict(size=15, family='Arial'),
    xaxis=dict(
        title='Hour of Day',
        tickmode='linear',
        tick0=0,
        dtick=1
    ),
    yaxis=dict(title='Avg Items in Basket'),
    template='plotly_white',
    height=500,
    width=950
)

fig_1b.show()
# BỎOOOOOOOOOOOOOOO

In [32]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Tạo subplot trục kép (Dual-axis)
fig_1c = make_subplots(specs=[[{"secondary_y": True}]])

# 1. Vẽ biểu đồ cột (Trục Y bên trái - Số lượng đơn)
fig_1c.add_trace(
    go.Bar(
        x=q6_pd['days_since_prior'],
        y=q6_pd['num_orders'],
        marker_color='#3B82F6',
        opacity=0.8,
        name='Orders Count',
        hovertemplate='<b>Days:</b> %{x}<br><b>Orders:</b> %{y:,.0f}<extra></extra>'
    ),
    secondary_y=False
)

# 2. Vẽ biểu đồ đường (Trục Y bên phải - % tích lũy)
fig_1c.add_trace(
    go.Scatter(
        x=q6_pd['days_since_prior'],
        y=q6_pd['cumulative_orders'] / q6_pd['cumulative_orders'].max() * 100,
        line=dict(color='#EF4444', width=2.5),
        name='Cumulative %',
        hovertemplate='<b>Days:</b> %{x}<br><b>Cumulative:</b> %{y:.1f}%<extra></extra>'
    ),
    secondary_y=True
)

# 3. Định cấu hình giao diện
fig_1c.update_layout(
    title='<b>Days Since Prior Order Distribution</b>',
    title_font=dict(size=15, family='Arial'),
    xaxis=dict(
        title='Days Since Prior Order',
        tickmode='linear',
        tick0=0,
        dtick=1
    ),
    yaxis=dict(title='Orders Count'),
    yaxis2=dict(title='Cumulative %'),
    template='plotly_white',
    height=500,
    width=950,
    legend=dict(
        orientation='h',
        yanchor='bottom', y=1.02,
        xanchor='right', x=1
    )
)

fig_1c.show()

In [33]:
import plotly.graph_objects as go

lower_bound = q18_pd['avg_basket_size'] - q18_pd['stddev_basket_size']
upper_bound = q18_pd['avg_basket_size'] + q18_pd['stddev_basket_size']

fig_1d = go.Figure()

# 1. Vẽ vùng bóng mờ biểu thị độ lệch chuẩn (StdDev Range)
fig_1d.add_trace(go.Scatter(
    x=q18_pd['order_number'], y=lower_bound,
    mode='lines', line=dict(width=0),
    showlegend=False, hoverinfo='skip'
))
fig_1d.add_trace(go.Scatter(
    x=q18_pd['order_number'], y=upper_bound,
    mode='lines', line=dict(width=0),
    fill='tonexty', fillcolor='rgba(16, 185, 129, 0.15)',
    showlegend=False, hoverinfo='skip'
))

# 2. Vẽ đường trung bình chính
fig_1d.add_trace(go.Scatter(
    x=q18_pd['order_number'],
    y=q18_pd['avg_basket_size'],
    mode='lines+markers',
    line=dict(color='#10B981', width=2.5),
    marker=dict(size=5),
    name='Avg Basket Size',
    hovertemplate='<b>Order #:</b> %{x}<br><b>Avg Items:</b> %{y:.2f} items<extra></extra>'
))

# 3. Cấu hình giao diện
fig_1d.update_layout(
    title='<b>Basket Size vs Order (with StdDev)</b>',
    title_font=dict(size=15, family='Arial'),
    xaxis=dict(title='Order Number'),
    yaxis=dict(title='Avg Items in Basket'),
    template='plotly_white',
    height=500,
    width=950
)

fig_1d.show()
# BỎOOOOOOOOOOOO

In [34]:
import plotly.graph_objects as go

q4_s = q4_pd.sort_values('num_customers', ascending=False)

seg_colors = ['#EF4444','#F59E0B','#10B981','#3B82F6'] # Đỏ, Cam, Lục, Lam

# 3. Vẽ biểu đồ Donut
fig_1e = go.Figure(data=go.Pie(
    labels=q4_s['customer_segment'],
    values=q4_s['num_customers'],
    hole=0.4,
    marker=dict(colors=seg_colors),
    textinfo='percent',
    sort=True,                  # Bắt buộc Plotly vẽ theo thứ tự giảm dần từ lớn đến bé
    direction='clockwise',      # Vẽ theo chiều kim đồng hồ bắt đầu từ đỉnh 12 giờ
    hovertemplate='<b>%{label}</b><br>Customers: %{value:,.0f}<br>% Share: %{percent}<extra></extra>'
))

# 4. Định cấu hình giao diện và chú thích (Legend)
fig_1e.update_layout(
    title='<b>Customer Segments</b>',
    title_font=dict(size=18, family='Arial'),
    template='plotly_white',
    height=550,
    width=850,
    legend=dict(
        orientation='v',        # Hiển thị chú thích dạng dọc gọn gàng
        yanchor='middle', 
        y=0.5,                  # Canh giữa biểu đồ theo chiều dọc
        xanchor='left', 
        x=1.05,                 # Đặt ngay bên phải biểu đồ Donut
        font=dict(size=15)
    )
)

fig_1e.show()
# Cân nhắc - suy nghĩ thử segment theo bubble chart

In [10]:
import plotly.graph_objects as go

# 1. Chuyển đổi dữ liệu mới từ Spark DataFrame sang Pandas
q19_pd = q19_equal_freq_df.toPandas()

# 2. Sắp xếp dữ liệu tự động dựa trên vị trí trung bình avg_position giảm dần
# (Vì Plotly vẽ từ dưới lên trên, sắp xếp giảm dần sẽ đưa nhóm đầu tiên "1. Very Early" lên trên cùng)
q19_s = q19_pd.sort_values('avg_position', ascending=False)

# 3. Vẽ biểu đồ cột ngang với màu sắc theo tỷ lệ Reorder Rate
fig_1f = go.Figure(data=go.Bar(
    x=q19_s['avg_reorder_rate_pct'],
    y=q19_s['cart_position_bucket'],
    orientation='h',
    marker=dict(
        color=q19_s['avg_reorder_rate_pct'],
        colorscale='rdbu_r'  # Thang màu từ Xanh dương (Thấp) sang Đỏ (Cao) cực kỳ trực quan
    ),
    # Đẩy tỉ lệ phân chia thực tế vào customdata để hiển thị xác thực mốc ~20%
    customdata=q19_s['pct_items_share'],
    hovertemplate='<b>%{y}</b><br>Share of Total Items: %{customdata:.2f}%<br>Reorder Rate: %{x:.2f}%<extra></extra>'
))

# 4. Định cấu hình giao diện tổng quan
fig_1f.update_layout(
    title='<b>Reorder Rate by Equal-Frequency Cart Position (Quintiles)</b>',
    title_font=dict(size=15, family='Arial'),
    xaxis=dict(title='Avg Reorder Rate %'),
    yaxis=dict(title='Equal-Frequency Cart Position'),
    template='plotly_white',
    height=500,
    width=950
)

fig_1f.show()

In [36]:
import plotly.graph_objects as go

# 1. Sắp xếp dữ liệu tăng dần theo số lượng đơn hàng trung bình
q13_s = q13_pd.sort_values('avg_orders', ascending=True)

# 2. Vẽ biểu đồ cột ngang phối màu trực quan
fig_1g = go.Figure(data=go.Bar(
    x=q13_s['avg_reorder_rate_pct'],
    y=q13_s['loyalty_tier'],
    orientation='h',
    marker=dict(
        color=['#3B82F6', '#10B981', '#F59E0B', '#EF4444'][:len(q13_s)] # Đồng bộ màu với bản cũ
    ),
    hovertemplate='<b>Loyalty Tier:</b> %{y}<br><b>Reorder Rate:</b> %{x:.2f}%<extra></extra>'
))

# 3. Định cấu hình giao diện
fig_1g.update_layout(
    title='<b>Reorder Rate by Loyalty Tier</b>',
    title_font=dict(size=15, family='Arial'),
    xaxis=dict(title='Avg Reorder Rate %'),
    yaxis=dict(title='Loyalty Tier'),
    template='plotly_white',
    height=500,
    width=950
)

fig_1g.show()
# Tạm thời bỏ qua

### **02. INSTACART – PRODUCT & CATEGORY ANALYSIS**

In [37]:
import plotly.graph_objects as go

# 1. Sắp xếp và sao chép dữ liệu để tránh ảnh hưởng tới DataFrame gốc
q2_s = q2_pd.sort_values('total_sold', ascending=True).copy()

# 2. Phân loại Organic bằng tìm kiếm chuỗi (Phòng vệ lỗi bằng na=False)
q2_s['is_organic'] = q2_s['product_name'].str.contains('organic', case=False, na=False)

# 3. Tách dữ liệu thành 2 phần để vẽ 2 trace riêng biệt nhằm tạo Legend tự động
organic_df = q2_s[q2_s['is_organic'] == True]
non_organic_df = q2_s[q2_s['is_organic'] == False]

fig_2a_opt2 = go.Figure()

# Trace 1: Nhóm sản phẩm Hữu cơ (Màu Xanh lá Emerald)
fig_2a_opt2.add_trace(go.Bar(
    x=organic_df['total_sold'],
    y=organic_df['product_name'],
    orientation='h',
    name='Organic Product',
    marker=dict(color='#10B981'),  # Màu lục Emerald
    hovertemplate='<b>%{y}</b> (Organic)<br>Total Sold: %{x:,.0f} units<extra></extra>'
))

# Trace 2: Nhóm sản phẩm Thông thường (Màu Đỏ Coral đối lập hoàn toàn)
fig_2a_opt2.add_trace(go.Bar(
    x=non_organic_df['total_sold'],
    y=non_organic_df['product_name'],
    orientation='h',
    name='Non-Organic Product',
    marker=dict(color='#EF4444'),  # Màu đỏ Coral tương phản mạnh [1]
    hovertemplate='<b>%{y}</b> (Non-Organic)<br>Total Sold: %{x:,.0f} units<extra></extra>'
))

# 4. Định cấu hình giao diện tổng quan
fig_2a_opt2.update_layout(
    title=dict(
        text='<b>Top 20 Best-Selling Products (Organic vs Non-Organic Classification)</b>',
        x=0.5,                       # Căn giữa tiêu đề chính
        font=dict(size=16, family='Arial')
    ),
    xaxis=dict(title='Total Sold (Units)'),
    yaxis=dict(
        title='Product Name', 
        tickfont=dict(size=9),
        categoryorder='array',
        categoryarray=q2_s['product_name'].tolist()
    ),
    barmode='overlay',
    template='plotly_white',
    height=650,
    width=1000,
    
    # ĐIỀU CHỈNH: Đưa bảng chú thích dàn ngang lên trên đồ thị
    legend=dict(
        orientation='h',             # Dàn các nhãn chú thích nằm ngang
        yanchor='bottom',
        y=1.03,                      # Nằm ngay trên viền trên của khung biểu đồ (y=1.0)
        xanchor='center',
        x=0.5,                       # Căn giữa theo chiều ngang
        font=dict(size=10)
    )
)

fig_2a_opt2.show()

In [38]:
import plotly.graph_objects as go

# 1. Sắp xếp dữ liệu tăng dần theo tỉ lệ mua lại
q3_s = q3_pd.sort_values('reorder_rate_pct', ascending=True)
mean_rate = q3_pd['reorder_rate_pct'].mean()

# 2. Vẽ biểu đồ cột ngang phân màu phân kỳ (Red-Yellow-Green)
fig_2b = go.Figure(data=go.Bar(
    x=q3_s['reorder_rate_pct'],
    y=q3_s['department'],
    orientation='h',
    marker=dict(
        color=q3_s['reorder_rate_pct'],
        colorscale='rdylgn'  # Hệ màu RdYlGn (Đỏ - Vàng - Lục) trực quan
    ),
    hovertemplate='<b>Department: %{y}</b><br>Reorder Rate: %{x:.2f}%<extra></extra>'
))

# 3. Thêm đường thẳng tham chiếu mức trung bình toàn hệ thống
fig_2b.add_vline(
    x=mean_rate,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Mean: {mean_rate:.1f}%",
    annotation_position="bottom right"
)

# 4. Định cấu hình giao diện
fig_2b.update_layout(
    title='<b>Reorder Rate by Department</b>',
    title_font=dict(size=15, family='Arial'),
    xaxis=dict(title='Reorder Rate %'),
    yaxis=dict(title='Department'),
    template='plotly_white',
    height=650,
    width=1100
)

fig_2b.show()

In [39]:
import plotly.express as px
import plotly.graph_objects as go

# 1. Sắp xếp dữ liệu
q7_s = q7_pd.sort_values('total_sold', ascending=True)

# 2. Lấy danh sách các phòng ban (department) xuất hiện trong biểu đồ
unique_departments = q7_s['department'].unique()

# ── SỬ DỤNG BẢNG MÀU TABLEAU 10 CHUYÊN NGHIỆP CỦA PLOTLY ──────────────────
# Bạn có thể dễ dàng đổi sang bảng màu khác bằng cách thay .T10 bằng:
# .D3 , .Safe , .Bold hoặc .Dark24 (nếu có quá nhiều phòng ban)
color_sequence = px.colors.qualitative.D3

# Ánh xạ động từng phòng ban với một màu trong bảng màu
dept_colors = {dept: color_sequence[i % len(color_sequence)] for i, dept in enumerate(unique_departments)}
# ─────────────────────────────────────────────────────────────────────────

fig_2c = go.Figure()

# 3. Vẽ từng trace riêng biệt
for dept in unique_departments:
    dept_df = q7_s[q7_s['department'] == dept]
    
    fig_2c.add_trace(go.Bar(
        x=dept_df['total_sold'],
        y=dept_df['aisle'],
        orientation='h',
        name=dept,
        marker=dict(color=dept_colors[dept]),
        hovertemplate='<b>Aisle: %{y}</b><br>Department: ' + dept + '<br>Total Sold: %{x:,.0f} units<extra></extra>'
    ))

# 4. Định cấu hình giao diện
fig_2c.update_layout(
    title=dict(
        text='<b>Top 15 Aisles by Sales</b>',
        x=0.5,
        font=dict(size=15, family='Arial')
    ),
    xaxis=dict(title='Total Sold (Units)'),
    yaxis=dict(
        title='Aisle',
        tickfont=dict(size=9),
        categoryorder='array',
        categoryarray=q7_s['aisle'].tolist()
    ),
    barmode='overlay',
    template='plotly_white',
    height=550,
    width=1000,
    legend=dict(
        title=dict(
            text='<b>Department</b>',
            font=dict(size=10)
        ),
        orientation='v',
        yanchor='middle',
        y=0.5,
        xanchor='left',
        x=1.02,
        font=dict(size=9)
    )
)

fig_2c.show()

In [40]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1. Chuẩn hóa dữ liệu tỷ lệ mua lại về dạng %
reorder_val = q8_pd['avg_reorder_rate'] * 100 if q8_pd['avg_reorder_rate'].max() <= 1.0 else q8_pd['avg_reorder_rate']

# Định dạng chuỗi hiển thị giá trị số liệu trên đầu mỗi cột
text_sold = [f"{val:.2f}M" for val in (q8_pd['total_items_sold'] / 1e6)]
text_reorder = [f"{val:.1f}%" for val in reorder_val]

# 2. Khởi tạo biểu đồ trục kép
fig_2d = make_subplots(specs=[[{"secondary_y": True}]])

# 3. Cột 1: Vẽ doanh số bán ra (Trục Y bên trái - Màu Xanh lá Emerald)
fig_2d.add_trace(
    go.Bar(
        x=q8_pd['product_type'],
        y=q8_pd['total_items_sold'] / 1e6,
        name='Items Sold (Millions)',
        marker_color='#10B981',  # Xanh lá Emerald
        offsetgroup=1,           # Gán nhóm cột số 1
        text=text_sold,          # Thêm text chú thích giá trị
        textposition='outside',  # Định vị chữ nằm phía ngoài (trên đầu cột)
        hovertemplate='<b>%{x}</b><br>Items Sold: %{y:.2f}M<extra></extra>'
    ),
    secondary_y=False
)

# 4. Cột 2: Vẽ tỷ lệ mua lại (Trục Y bên phải - Màu Cam Accent)
fig_2d.add_trace(
    go.Bar(
        x=q8_pd['product_type'],
        y=reorder_val,
        name='Reorder Rate %',
        marker_color='#F59E0B',  # Màu cam tương phản mạnh
        offsetgroup=2,           # Gán nhóm cột số 2 để đẩy cột nằm cạnh nhau
        text=text_reorder,       # Thêm text chú thích giá trị
        textposition='outside',  # Định vị chữ nằm phía ngoài (trên đầu cột)
        hovertemplate='<b>%{x}</b><br>Reorder Rate: %{y:.2f}%<extra></extra>'
    ),
    secondary_y=True
)

# 5. Cấu hình giao diện tổng quan và kích hoạt chế độ cột kép
fig_2d.update_layout(
    title=dict(
        text='<b>Organic vs Non-Organic Analysis</b>',
        x=0.5,
        font=dict(size=15, family='Arial')
    ),
    xaxis=dict(title='Product Type'),
    
    # Trục Y bên trái: Tăng nhẹ giới hạn trên (15%) để tạo khoảng trống cho nhãn chữ M
    yaxis=dict(
        title='Items Sold (Millions)',
        range=[0, (q8_pd['total_items_sold'] / 1e6).max() * 1.15]
    ),
    
    # Trục Y bên phải: Tăng nhẹ giới hạn trên (15%) để tạo khoảng trống cho nhãn chữ %
    yaxis2=dict(
        title='Avg Reorder Rate %', 
        overlaying='y', 
        side='right',
        range=[0, reorder_val.max() * 1.15]
    ),
    
    barmode='group',             # Kích hoạt chế độ cột kép (side-by-side)
    template='plotly_white',
    height=500,
    width=850,
    
    # Định dạng chú giải nằm ngang phía trên
    legend=dict(
        orientation='h',
        yanchor='bottom', y=1.02,
        xanchor='right', x=1
    )
)

fig_2d.show()

In [41]:
import plotly.graph_objects as go

# 1. Sắp xếp dữ liệu theo tỉ lệ mua lại tăng dần
q11_s = q11_pd.sort_values('reorder_pct', ascending=True)

fig_2e = go.Figure()

# 2. Vẽ phân khúc Mua lần đầu (First Buy %)
fig_2e.add_trace(go.Bar(
    x=q11_s['first_buy_pct'],
    y=q11_s['department'],
    orientation='h',
    name='First Buy %',
    marker_color='#3B82F6',  # PALETTE_SEQ
    hovertemplate='<b>%{y}</b><br>First Buy: %{x:.2f}%<extra></extra>'
))

# 3. Vẽ phân khúc Mua lặp lại (Reorder %) xếp chồng tiếp nối
fig_2e.add_trace(go.Bar(
    x=q11_s['reorder_pct'],
    y=q11_s['department'],
    orientation='h',
    name='Reorder %',
    marker_color='#EF4444',  # PALETTE_RED
    hovertemplate='<b>%{y}</b><br>Reorder: %{x:.2f}%<extra></extra>'
))

# 4. Định cấu hình giao diện (Chế độ barmode='stack' và tăng chiều cao lên 650px)
fig_2e.update_layout(
    title='<b>First Buy vs Reorder Percentage by Department</b>',
    title_font=dict(size=15, family='Arial'),
    xaxis=dict(title='Percentage %', range=[0, 100]),
    yaxis=dict(title='Department'),
    barmode='stack',  # Cấu hình quan trọng giúp xếp chồng 2 phần bar ngang
    template='plotly_white',
    height=650,
    width=1000,
    legend=dict(
        orientation='h',
        yanchor='bottom', y=1.02,
        xanchor='right', x=1
    )
)

fig_2e.show()
# Last Choice

In [42]:
import plotly.graph_objects as go

# 1. Sắp xếp dữ liệu theo tổng số lượng bán ra tăng dần
# (Giúp phòng ban có quy mô doanh số lớn nhất nằm ở trên cùng)
q11_s = q11_pd.sort_values('total', ascending=True)

fig_2e = go.Figure()

# 2. Vẽ phân khúc Số lượng Mua lần đầu (First Buy Volume)
fig_2e.add_trace(go.Bar(
    x=q11_s['first_time_buys'],
    y=q11_s['department'],
    orientation='h',
    name='First Buy Volume',
    marker_color='#3B82F6',  # Xanh lam
    hovertemplate='<b>%{y}</b><br>First Buy: %{x:,.0f} units<extra></extra>'
))

# 3. Vẽ phân khúc Số lượng Mua lặp lại (Reorder Volume) xếp chồng tiếp nối
fig_2e.add_trace(go.Bar(
    x=q11_s['reorders'],
    y=q11_s['department'],
    orientation='h',
    name='Reorder Volume',
    marker_color='#EF4444',  # Đỏ
    hovertemplate='<b>%{y}</b><br>Reorder: %{x:,.0f} units<extra></extra>'
))

# 4. Định cấu hình giao diện ở chế độ xếp chồng thường (barmode='stack')
fig_2e.update_layout(
    title='<b>First Buy vs Reorder Volume by Department</b>',
    title_font=dict(size=15, family='Arial'),
    xaxis=dict(title='Total Items Sold (Units)'),
    yaxis=dict(title='Department'),
    barmode='stack',  # Cấu hình xếp chồng cột thông thường
    template='plotly_white',
    height=650,
    width=1000,
    legend=dict(
        orientation='h',
        yanchor='bottom', y=1.02,
        xanchor='right', x=1
    )
)

fig_2e.show()

### **03. INSTACART – PRODUCT & CATEGORY ANALYSIS**

In [44]:
import plotly.graph_objects as go

# 1. Sắp xếp và trích xuất dữ liệu (Giữ nguyên top 15)
q12_pd['pair_label'] = (q12_pd['product_a'].str[:22] + ' × ' + q12_pd['product_b'].str[:22])
q12_s = q12_pd.sort_values('co_occurrence').tail(15).copy()

# 2. Kiểm tra xem cặp sản phẩm có chứa "Banana" hay không (Không phân biệt hoa/thường)
q12_s['has_banana'] = (
    q12_s['product_a'].str.contains('banana', case=False, na=False) | 
    q12_s['product_b'].str.contains('banana', case=False, na=False)
)

# 3. Tách dữ liệu thành 2 nhóm để tạo Legend phân loại rõ ràng
banana_df = q12_s[q12_s['has_banana'] == True]
nobanana_df = q12_s[q12_s['has_banana'] == False]

fig_3a_opt1 = go.Figure()

# Trace 1: Các kết hợp hiển nhiên với Chuối (Màu Xám Nhạt - Muted Gray)
fig_3a_opt1.add_trace(go.Bar(
    x=banana_df['co_occurrence'],
    y=banana_df['pair_label'],
    orientation='h',
    name='Common Banana Associations (Muted)',
    marker=dict(color='#D1D5DB'),  # Xám nhạt để giảm bớt sự chú ý
    hovertemplate='<b>Pair: %{y}</b><br>Co-occurrences: %{x:,.0f} times<extra></extra>'
))

# Trace 2: Kết hợp ẩn không chứa Chuối (Màu Xanh Dương Đậm - Highlight Blue)
fig_3a_opt1.add_trace(go.Bar(
    x=nobanana_df['co_occurrence'],
    y=nobanana_df['pair_label'],
    orientation='h',
    name='Hidden Cross-Sell Insights (Highlighted)',
    marker=dict(color='#1D4ED8'),  # Xanh dương đậm nổi bật thu hút ánh nhìn
    hovertemplate='<b>Pair: %{y}</b><br>Co-occurrences: %{x:,.0f} times<extra></extra>'
))

# 4. Định cấu hình Layout
fig_3a_opt1.update_layout(
    title=dict(
        text='<b>Top 15 Product Pairs (Highlighting Hidden Cross-Sell Insights)</b>',
        x=0.5,
        font=dict(size=15, family='Arial')
    ),
    xaxis=dict(title='Co-occurrence Count'),
    yaxis=dict(
        title='Product Pair', 
        tickfont=dict(size=9),
        categoryorder='array',
        categoryarray=q12_s['pair_label'].tolist()
    ),
    barmode='overlay',
    template='plotly_white',
    height=650,
    width=1200,
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.03,
        xanchor='center',
        x=0.5,
        font=dict(size=10)
    )
)

fig_3a_opt1.show()
# Bỏ

In [45]:
import plotly.graph_objects as go

# 1. Sắp xếp và trích xuất dữ liệu (Giữ nguyên top 15)
q12_pd['pair_label'] = (q12_pd['product_a'].str[:22] + ' × ' + q12_pd['product_b'].str[:22])
q12_s = q12_pd.sort_values('co_occurrence').tail(15).copy()

# 2. Kiểm tra xem CẢ HAI sản phẩm trong cặp có phải là Organic hay không
q12_s['is_org_a'] = q12_s['product_a'].str.contains('organic', case=False, na=False)
q12_s['is_org_b'] = q12_s['product_b'].str.contains('organic', case=False, na=False)
q12_s['both_organic'] = q12_s['is_org_a'] & q12_s['is_org_b']

# 3. Tách dữ liệu thành 2 nhóm để tạo Legend
organic_pairs_df = q12_s[q12_s['both_organic'] == True]
mixed_pairs_df = q12_s[q12_s['both_organic'] == False]

fig_3a_opt2 = go.Figure()

# Trace 1: Cặp thuần hữu cơ (Màu Xanh Lá Emerald)
fig_3a_opt2.add_trace(go.Bar(
    x=organic_pairs_df['co_occurrence'],
    y=organic_pairs_df['pair_label'],
    orientation='h',
    name='Organic × Organic Pair',
    marker=dict(color='#10B981'),  # Xanh lá Emerald (Biểu trưng cho lối sống xanh)
    hovertemplate='<b>Pair: %{y}</b> (Organic)<br>Co-occurrences: %{x:,.0f} times<extra></extra>'
))

# Trace 2: Cặp hỗn hợp hoặc thông thường (Màu Xám Slate)
fig_3a_opt2.add_trace(go.Bar(
    x=mixed_pairs_df['co_occurrence'],
    y=mixed_pairs_df['pair_label'],
    orientation='h',
    name='Standard / Mixed Pair',
    marker=dict(color='#9CA3AF'),  # Xám Slate trung tính
    hovertemplate='<b>Pair: %{y}</b> (Mixed/Standard)<br>Co-occurrences: %{x:,.0f} times<extra></extra>'
))

# 4. Định cấu hình Layout
fig_3a_opt2.update_layout(
    title=dict(
        text='<b>Top 15 Product Pairs (Organic vs Standard Basket Structure)</b>',
        x=0.5,
        font=dict(size=15, family='Arial')
    ),
    xaxis=dict(title='Co-occurrence Count'),
    yaxis=dict(
        title='Product Pair', 
        tickfont=dict(size=9),
        categoryorder='array',
        categoryarray=q12_s['pair_label'].tolist()
    ),
    barmode='overlay',
    template='plotly_white',
    height=650,
    width=1200,
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=1.03,
        xanchor='center',
        x=0.5,
        font=dict(size=10)
    )
)

fig_3a_opt2.show()

In [46]:
import plotly.graph_objects as go

# 1. Chuẩn hóa dữ liệu tỉ lệ mua lại về dạng %
reorder_val = q18_pd['avg_reorder_pct'] * 100 if q18_pd['avg_reorder_pct'].max() <= 1.0 else q18_pd['avg_reorder_pct']
mean_reorder = reorder_val.mean()

# 2. Vẽ biểu đồ đường marker vuông tông màu cam ấm áp
fig_3b = go.Figure(data=go.Scatter(
    x=q18_pd['order_number'],
    y=reorder_val,
    mode='lines+markers',
    line=dict(color='#F59E0B', width=2.5),  # Màu cam đất của Palette
    marker=dict(size=6, symbol='square'),
    name='Reorder Rate %',
    hovertemplate='Order Sequence: #%{x}<br>Avg Reorder Rate: %{y:.2f}%<extra></extra>'
))

# 3. Thêm đường thẳng ngang tham chiếu tỉ lệ trung bình toàn hệ thống
fig_3b.add_hline(
    y=mean_reorder,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Mean: {mean_reorder:.1f}%",
    annotation_position="top left"
)

# 4. Định cấu hình giao diện
fig_3b.update_layout(
    title='<b>Reorder Rate vs Order Sequence</b>',
    title_font=dict(size=15, family='Arial'),
    xaxis=dict(title='Order Number'),
    yaxis=dict(title='Avg Reorder %'),
    template='plotly_white',
    height=600,
    width=1050
)

fig_3b.show()

In [47]:
import plotly.graph_objects as go

# Lấy thông tin tên sản phẩm làm nhãn hiển thị khi di chuột (hover)
hover_text = q10_pd['product_name'] if 'product_name' in q10_pd.columns else q10_pd.index

# Vẽ biểu đồ Scatter kèm thanh màu (Colorbar) hiển thị ở biên phải tiêu chuẩn
fig_3c = go.Figure(data=go.Scatter(
    x=q10_pd['total_orders'],
    y=q10_pd['reorder_rate_pct'],
    mode='markers',
    text=hover_text,
    marker=dict(
        size=10,
        color=q10_pd['unique_buyers'],
        colorscale='bupu',  # Sử dụng bảng màu bupu chuẩn của Plotly
        showscale=True,
        colorbar=dict(
            title='Unique Buyers',
            thickness=15
        )
    ),
    hovertemplate='<b>%{text}</b><br>Total Orders: %{x:,.0f}<br>Reorder Rate: %{y:.2f}%<br>Buyers: %{marker.color:,.0f}<extra></extra>'
))

# Định cấu hình giao diện
fig_3c.update_layout(
    title='<b>Hidden Gems (High Reorder, Niche Volume)</b>',
    title_font=dict(size=15, family='Arial'),
    xaxis=dict(title='Total Orders'),
    yaxis=dict(title='Reorder Rate %'),
    template='plotly_white',
    height=500,
    width=1000
)

fig_3c.show()

In [52]:
import plotly.graph_objects as go

# 1. Trực quan hóa dữ liệu chéo (Pivot table) ban đầu
pivot14 = q14_pd.pivot(index='department', columns='time_slot', values='items_sold').fillna(0)

# 2. TIẾN HÀNH SẮP XẾP LẠI BIỂU ĐỒ HEATMAP
# - Trục X: Sắp xếp theo chu kỳ thời gian tự nhiên (Sáng -> Chiều -> Tối -> Đêm)
time_order = ['06-11 Sáng', '12-17 Chiều', '18-21 Tối', '22-05 Đêm/Khuya']
pivot14 = pivot14.reindex(columns=[t for t in time_order if t in pivot14.columns])

# - Trục Y: Sắp xếp các phòng ban theo tổng doanh số bán ra (items_sold) tăng dần
# (Do Plotly vẽ trục Y từ dưới lên, nên việc sắp xếp tăng dần giúp đưa phòng ban bán chạy nhất lên trên cùng)
dept_order_sales = q14_pd.groupby('department')['items_sold'].sum().sort_values(ascending=True).index
pivot14 = pivot14.reindex(index=dept_order_sales)

# 💡 Mẹo nhỏ: Nếu bạn muốn sắp xếp theo thứ hạng rnk trung bình (Thứ hạng càng tốt/nhỏ thì nằm trên cùng)
# bạn có thể dùng dòng code bên dưới thay thế cho dòng `dept_order_sales`:
# dept_order_rnk = q14_pd.groupby('department')['rnk'].mean().sort_values(ascending=False).index
# pivot14 = pivot14.reindex(index=dept_order_rnk)


# 3. Vẽ biểu đồ Heatmap tông màu Blues có thanh nhiệt bên cạnh
fig_3d = go.Figure(data=go.Heatmap(
    z=pivot14.values,
    x=pivot14.columns,
    y=pivot14.index,     # Trục Y đã được sắp xếp khoa học
    colorscale='blues',
    showscale=True,
    colorbar=dict(
        title='Items Sold',
        thickness=15
    ),
    hovertemplate='<b>Department: %{y}</b><br>Time Slot: %{x}<br>Items Sold: %{z:,.0f} units<extra></extra>'
))

# 4. Định cấu hình giao diện
fig_3d.update_layout(
    title='<b>Dept Sales by Time Slot (Sorted by Sales Volume)</b>',
    title_font=dict(size=15, family='Arial'),
    xaxis=dict(title='Time Slot'),
    yaxis=dict(title='Department', tickfont=dict(size=8)),
    template='plotly_white',
    height=550,
    width=950
)

fig_3d.show()
# chia lại khung giờ

In [49]:
import plotly.graph_objects as go

# 1. Trích xuất top 15 phòng ban có doanh số lớn nhất
q20_b = q20_pd.sort_values('total_items_sold', ascending=False).head(15)

# 2. Co giãn kích thước bong bóng (max size = 50 tương thích tốt biểu đồ độc lập)
sizes = q20_b['num_products']
scaled_sizes = (sizes / sizes.max() * 50).clip(lower=10)

# 3. Vẽ biểu đồ bong bóng kèm nhãn tên văn bản phía trên bong bóng
fig_3e = go.Figure(data=go.Scatter(
    x=q20_b['avg_reorder_rate_pct'],
    y=q20_b['avg_sold_per_product'],
    mode='markers+text',
    text=q20_b['department'],
    textposition='top center',
    marker=dict(
        size=scaled_sizes,
        color=q20_b['total_items_sold'],
        colorscale='viridis',
        showscale=True,
        colorbar=dict(
            title='Total Items Sold',
            thickness=15
        )
    ),
    hovertemplate='<b>Department: %{text}</b><br>Avg Reorder: %{x:.2f}%<br>Avg Sold/Product: %{y:,.1f}<br>Products Count: %{marker.size}<extra></extra>'
))

# 4. Định cấu hình giao diện
fig_3e.update_layout(
    title='<b>Dept Portfolio Map (bubble size = #products)</b>',
    title_font=dict(size=15, family='Arial'),
    xaxis=dict(title='Avg Reorder Rate %'),
    yaxis=dict(title='Avg Sales per Product'),
    template='plotly_white',
    height=550,
    width=1000
)

fig_3e.show()